In [1]:
import pandas as pd

# Convert CSV files into dataframes
pairwise_df = pd.read_csv('../data/sample/sample-pairwise.csv')
product_df = pd.read_csv('../data/intermediary/product-info.csv')

In [6]:
pairwise_df.dtypes
pairwise_df[['P_i','P_j','P_ij']].head(10)


,P_i,P_j,P_ij
0,0.000057,0.006034,3.110542e-07
1,0.000101,0.006034,3.110542e-07
2,0.000146,0.006034,3.110542e-07
3,0.000245,0.006034,3.110542e-07
4,0.000124,0.006034,3.110542e-07
5,0.000050,0.006034,3.110542e-07
6,0.000066,0.006034,3.110542e-07
7,0.000464,0.006034,3.110542e-07
8,0.000095,0.006034,3.110542e-07
9,0.000132,0.006034,3.110542e-07


In [23]:
def compute_lift(product_df, pairwise_df, output_csv=None):
    if output_csv:
        with open(output_csv, "w") as f:
            f.write("product_i,product_j,lift, P_ij\n")

    pairwise_df_i = pairwise_df.groupby("product_i")
    pairwise_df_j = pairwise_df.groupby("product_j")

    for row in product_df.itertuples(index=False):
        product_id = row.product_id

        product_probs = pairwise_df_i.get_group(product_id) if product_id in pairwise_df_i.groups else pd.DataFrame()
        product_j_probs = pairwise_df_j.get_group(product_id) if product_id in pairwise_df_j.groups else pd.DataFrame()

        # Reverse relationships
        products_rev = product_j_probs.rename(columns={
            'product_i': 'product_j',
            'product_j': 'product_i',
            'P_i': 'P_j',
            'P_j': 'P_i'
        })

        product_probs_df = pd.concat([product_probs, products_rev], ignore_index=True)
        if product_probs_df.empty:
            continue

        # Compute metrics
        product_probs_df['lift'] = product_probs_df.apply(lambda x: (x.P_ij / (x.P_i * x.P_j) if x.P_i*x.P_j > 0 else 0), axis=1)
        

        # Rename columns before saving
        pairwise_lift_df = product_probs_df.copy()
        pairwise_lift_df = pairwise_lift_df[['product_i', 'product_j', 'lift', 'P_ij']]

        # Append to CSV
        if output_csv:
            pairwise_lift_df.to_csv(output_csv, mode='a', index=False, header=False)
        else:
            return pairwise_lift_df  # For testing

    print(f"Completed lift calculations. Saved to {output_csv if output_csv else 'DataFrame'}")

compute_lift(product_df, pairwise_df, output_csv="../data/sample/obj3/sample-lift.csv")

Completed lift calculations. Saved to ../data/sample/obj3/sample-lift.csv


In [5]:
pairwise_lift_df = pd.read_csv('../data/sample/obj3/sample-lift.csv')
print(pairwise_lift_df.describe())
print(pairwise_lift_df.columns.tolist())

          product_i     product_j          lift          P_ij
count  2.902857e+07  2.902857e+07  2.902857e+07  2.902857e+07
mean   2.501245e+04  2.501245e+04  4.230540e+01  1.256727e-06
std    1.427098e+04  1.427098e+04  2.908702e+02  3.576387e-06
min    1.000000e+00  1.000000e+00 -3.617889e+03 -3.981494e-05
25%    1.260600e+04  1.260600e+04  1.737986e+00  3.110542e-07
50%    2.522300e+04  2.522300e+04  4.597132e+00  3.110542e-07
75%    3.722900e+04  3.722900e+04  1.593099e+01  9.331625e-07
max    4.968800e+04  4.968800e+04  6.429748e+04  3.950388e-05
['product_i', 'product_j', 'lift', ' P_ij']


In [51]:
import networkx as nx
import community as community_louvain
import pandas as pd

# Convert CSV files into dataframes
pairwise_lift_df = pd.read_csv('../data/sample/obj3/sample-lift.csv')

def build_complement_network(pairwise_lift_df):

    # Filter significant edges
    #lift_threshold = 5  # slightly above median
    #support_threshold = 1e-6  # roughly between 50th and 75th percentile
    lift_threshold = 3  
    support_threshold = 5e-7  
    pairwise_lift_df = pairwise_lift_df[
        (pairwise_lift_df['lift'] > lift_threshold) &
        (pairwise_lift_df[' P_ij'] > support_threshold)
    ]


    # Build complement network

    # Ensure only positive lift values
    pairwise_lift_df = pairwise_lift_df.dropna(subset=['lift'])
    pairwise_lift_df['lift'] = pairwise_lift_df['lift'].clip(lower=0)

    G = nx.Graph()
    for _, row in pairwise_lift_df.iterrows():
        G.add_edge(
            row['product_i'],
            row['product_j'],
            weight=row['lift']  # could also use lift * cond_prob_i_to_j
        )

    # Apply Louvain algorithm to detect clusters
    try:
        partition = community_louvain.best_partition(G, weight='weight')
        nx.set_node_attributes(G, partition, 'cluster')
    except ImportError:
        print("Louvain community detection not installed; skipping cluster assignment.")
        partition = None

    # Identify top complements for each product
    top_n = 5
    complements = []

    product_ids = set(pairwise_lift_df['product_i']).union(set(pairwise_lift_df['product_j']))

    for product in product_ids:
        if product not in G:
            continue
        neighbors = G[product]
        ranked = sorted(
            [(nbr, G[product][nbr]['weight']) for nbr in neighbors],
            key=lambda x: x[1],
            reverse=True
        )
        for nbr, weight in ranked[:top_n]:
            complements.append({
                'product_id': int(product),
                'complement_id': int(nbr),
                'lift': weight,
                'cluster': partition[product] if partition else None
            })

    complements_df = pd.DataFrame(complements)
    return complements_df

complements_df = build_complement_network(pairwise_lift_df)
complements_df.to_csv("../data/sample/obj3/sample-complements-top5.csv", mode='a', index=False, header=True)

In [ ]:
# Validation
orders_test_df = pd.read_csv('../dataset/order_products__train.csv')

def validate_complements_in_test(complements_df, test_df):

    # Self-join test orders on order_id to get all product pairs in the same order
    pairs_in_test = (
        test_df
        .merge(test_df, on="order_id")
        .query("product_id_x != product_id_y")[["product_id_x", "product_id_y"]]
        .drop_duplicates()
    )

    # Rename for clarity
    pairs_in_test.columns = ["product_id", "complement_id"]

    # Merge with your predicted complements to check which pairs appear in test
    observed_complements = complements_df.merge(
        pairs_in_test,
        on=["product_id", "complement_id"],
        how="inner"
    )

    print(f"{len(observed_complements)} complement pairs found in test data.")
    precision = (len(observed_complements) / len(complements_df)) * 100
    print(f"Precision: {precision:.2f} of predicted complements that co-occurred in the test set")

top10_df = pd.read_csv('../data/sample/obj3/sample-complements-top10.csv')
print("Top 10 Complements:")
validate_complements_in_test(top10_df, orders_test_df)

top5_df = pd.read_csv('../data/sample/obj3/sample-complements-top5.csv')
print("Top 5 Complements:")
validate_complements_in_test(top5_df, orders_test_df)


C:\Users\jenle\AppData\Local\Temp\ipykernel_17084\1375373350.py:28: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  top10_df = pd.read_csv('../data/sample/obj3/sample-complements-top10.csv')


Top 10 Complements:
88934 complement pairs found in test data.
Precision: 6.02 of predicted complements that co-occurred in the test set


C:\Users\jenle\AppData\Local\Temp\ipykernel_17084\1375373350.py:32: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  top5_df = pd.read_csv('../data/sample/obj3/sample-complements-top5.csv')


Top 5 Complements:
17514 complement pairs found in test data.
Precision: 8.68 of predicted complements that co-occurred in the test set
